In [34]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [35]:
import pandas as pd
import numpy as np
df = pd.read_csv('fashion-mnist_train.csv')
df.head(3)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0


In [36]:
from sklearn.model_selection import train_test_split
X = df.drop('label', axis=1)
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [37]:
X_train = X_train/255
X_test = X_test/255

In [38]:
X_train = X_train.to_numpy()
X_test = X_test.to_numpy()
y_train = y_train.to_numpy()
y_test = y_test.to_numpy()

In [39]:
import torch
X_train = torch.from_numpy(X_train).to(torch.float32)
X_test = torch.from_numpy(X_test).to(torch.float32)
y_train = torch.from_numpy(y_train).to(torch.long)
y_test = torch.from_numpy(y_test).to(torch.long)

In [40]:
from torch.utils.data import Dataset, DataLoader
class CustomDataset(Dataset):
  def __init__(self, features, labels):
    self.features = features.reshape(-1, 1, 28, 28)
    self.labels = labels

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):
    return self.features[index], self.labels[index]

In [41]:
train_ds = CustomDataset(X_train, y_train)
test_ds = CustomDataset(X_test, y_test)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, pin_memory=True)

In [46]:
import torch
import torch.nn as nn

class Neural_Network(nn.Module):
  def __init__(self, num_features):
    super().__init__()
    self.features = nn.Sequential(
        # Conv layer 1 with 32 filters
        nn.Conv2d(num_features, 32, kernel_size=3, padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(32),
        nn.MaxPool2d(kernel_size=2, stride=2),

        # Conv layer 2 with 64 filters
        nn.Conv2d(32, 64, kernel_size=3, padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(64),
        nn.MaxPool2d(kernel_size=2, stride=2)
    )
    self.classifier = nn.Sequential(
        # ANN layer
        nn.Flatten(),
        nn.Linear(64*7*7, 128),
        nn.ReLU(),
        nn.Dropout(p=0.4),
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Dropout(p=0.4),
        nn.Linear(64, 10)
    )

  def forward(self, num_features):
    x = self.features(num_features)
    x = self.classifier(x)
    return x

In [49]:
learning_rate = 0.1
epochs = 40

In [50]:
model = Neural_Network(1)
model = model.to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay=1e-4)

for epoch in range(epochs):
  for batch_features, batch_labels in train_loader:
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)
    y_pred = model(batch_features)
    loss = loss_function(y_pred, batch_labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  print(f'Epoch: {epoch+1}, Loss:{loss.item()}')

Epoch: 1, Loss:0.6423704624176025
Epoch: 2, Loss:0.16706207394599915
Epoch: 3, Loss:0.22681832313537598
Epoch: 4, Loss:0.2299051433801651
Epoch: 5, Loss:0.2171764075756073
Epoch: 6, Loss:0.2263461947441101
Epoch: 7, Loss:0.2457519918680191
Epoch: 8, Loss:0.11853218078613281
Epoch: 9, Loss:0.3252626359462738
Epoch: 10, Loss:0.17031477391719818
Epoch: 11, Loss:0.03820148855447769
Epoch: 12, Loss:0.4003296494483948
Epoch: 13, Loss:0.12067756801843643
Epoch: 14, Loss:0.1639942228794098
Epoch: 15, Loss:0.11782003939151764
Epoch: 16, Loss:0.24405576288700104
Epoch: 17, Loss:0.11271858215332031
Epoch: 18, Loss:0.04955786466598511
Epoch: 19, Loss:0.01894526369869709
Epoch: 20, Loss:0.41731616854667664
Epoch: 21, Loss:0.12555287778377533
Epoch: 22, Loss:0.20866942405700684
Epoch: 23, Loss:0.16621661186218262
Epoch: 24, Loss:0.2720191776752472
Epoch: 25, Loss:0.18806499242782593
Epoch: 26, Loss:0.2215156853199005
Epoch: 27, Loss:0.07593613862991333
Epoch: 28, Loss:0.07149426639080048
Epoch: 29, 

In [51]:
# Train accuracy
model.eval()
total = 0
correct = 0

with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)
    y_pred = model(batch_features)
    _, predicted = torch.max(y_pred, 1)
    total += batch_labels.shape[0]
    correct += (predicted == batch_labels).sum().item()
print(f'Accuracy: {correct/total}')

Accuracy: 0.9233333333333333
